In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from transformers import (
    BertTokenizer,
    BertForMaskedLM,
    BertConfig,
    LineByLineTextDataset,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
import torch
from sklearn.model_selection import train_test_split
from typing import Literal
from torch import nn
from transformers import BertModel,AutoModel
from huggingface_hub import hf_hub_download
from typing import Literal
import json

c:\Users\ritik\anaconda3\envs\datascience310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nbs_ = pd.read_csv('E:\\GitHub\\Multi-Model-Bias-Detection-and-Debiasing-the-News\\EDA\\No_Corrupted_NBS.csv')

In [3]:
nbs_

,unique_id,outlet_x,title,content,image_description_x,text_label_x,multimodal_label_x,outlet_y,headline,top_image,article_text,new_categories,news_categories_confidence_scores,text_label_y,multimodal_label_y,image_description_y,image_filename,image_path
0,23400064a2,CBC.ca,Anishnabeg Outreach develops self-serve mental...,A new resource developed by Anishnabeg Outreac...,The image shows a person standing in front of ...,Unlikely,Likely,CBC.ca,Anishnabeg Outreach develops self-serve mental...,"{'bytes': None, 'path': '/projects/NMB-Plus/ra...",A new resource developed by Anishnabeg Outreac...,['Health' 'Local/Regional'],[0.85 0.75],Unlikely,Likely,The image shows a person standing in front of ...,23400064a2.jpeg,C:\Users\ritik\Downloads\images\images\2340006...
1,0bad329c25,CBC.ca,South Asian newcomers to Canada say online hat...,International student Miran Kadri had many thi...,Two individuals with backpacks walking on a ci...,Likely,Unlikely,CBC.ca,South Asian newcomers to Canada say online hat...,"{'bytes': None, 'path': '/projects/NMB-Plus/ra...",International student Miran Kadri had many thi...,['National' 'Opinion/Editorial'],[0.85 0.65],Likely,Unlikely,Two individuals with backpacks walking on a ci...,0bad329c25.jpeg,C:\Users\ritik\Downloads\images\images\0bad329...
2,267f8bc361,CBC.ca,Program that pairs nurses with RCMP on mental ...,A program in Fort McMurray that pairs police o...,A smiling woman in a pink shirt is seated in a...,Unlikely,Likely,CBC.ca,Program that pairs nurses with RCMP on mental ...,"{'bytes': None, 'path': '/projects/NMB-Plus/ra...",A program in Fort McMurray that pairs police o...,['Local/Regional' 'Health'],[0.85 0.75],Unlikely,Likely,A smiling woman in a pink shirt is seated in a...,267f8bc361.jpeg,C:\Users\ritik\Downloads\images\images\267f8bc...
3,2fab02aa42,CBC.ca,Should social media come with a health warning...,The U.S. surgeon general has called for social...,"A person is holding a smartphone, possibly rea...",Likely,Unlikely,CBC.ca,Should social media come with a health warning...,"{'bytes': None, 'path': '/projects/NMB-Plus/ra...",The U.S. surgeon general has called for social...,['Health' 'Opinion/Editorial'],[0.9 0.7],Likely,Unlikely,"A person is holding a smartphone, possibly rea...",2fab02aa42.jpeg,C:\Users\ritik\Downloads\images\images\2fab02a...
4,004798d706,CBC.ca,Inside Out 2: What the movie’s 4 new emotions ...,"Emotions are clues about ourselves, says exper...",A character with blue hair and a green dress s...,Likely,Unlikely,CBC.ca,Inside Out 2: What the movie’s 4 new emotions ...,"{'bytes': None, 'path': '/projects/NMB-Plus/ra...","Emotions are clues about ourselves, says exper...",['Entertainment' 'Health'],[0.85 0.75],Likely,Unlikely,A character with blue hair and a green dress s...,004798d706.jpeg,C:\Users\ritik\Downloads\images\images\004798d...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6244,0471e108dd,CNN,Opinion: We Germans are making Trump ‘thunders...,Editor’s Note: Anna Sauerbrey is the foreign e...,A black and white photo of a large assembly ha...,Likely,Likely,CNN,Opinion: We Germans are making Trump ‘thunders...,"{'bytes': None, 'path': '/projects/NMB-Plus/wh...",Editor’s Note: Anna Sauerbrey is the foreign e...,['Opinion/Editorial' 'Politics'],[0.9 0.8],Likely,Likely,A black and white photo of a large assembly ha...,0471e108dd.jpg,C:\Users\ritik\Downloads\images\images\0471e10...
6245,1e9e145f6e,CNN,Brazil’s floods smashed through barriers desig...,Karine Pitana had just paid off the last insta...,An aerial view of a flooded road with a yellow...,Likely,Likely,CNN,Brazil’s floods smashed through barriers desig...,"{'bytes': None, 'path': '/projects/NMB-Plus/wh...",Karine Pitana had just paid off the last insta...,['Local/Regional' 'National'],[0.85 0.75],Likely,Likely,An aerial view of a flooded road with a yellow...,1e9e145f6e.jpg,C:\Users\ritik\Downloads\images\images\1e9e145...
6246,0bc2d5dd25,CNN,"In a city cut off from the w

In [4]:
nbs_['text_labels']=nbs_['text_label_x'].apply(lambda x : 'biased' if x=='Likely' else 'non-biased')

In [5]:
nbs_['MultiModal_Label'] =nbs_['multimodal_label_x']
nbs_=nbs_.drop(columns=['multimodal_label_x','multimodal_label_y'])

In [6]:
nbs_['caption']=nbs_['image_description_x']
nbs_=nbs_.drop(columns=['image_description_x','image_description_y'])



In [7]:
'''Removing non words '''

nbs_['article_text'] = nbs_['article_text'].replace(to_replace=r'[^\w\s]', value='', regex=True)
nbs_['caption'] = nbs_['caption'].replace(to_replace=r'[^\w\s]', value='', regex=True)

'''Lower casing the text'''

nbs_ = nbs_.applymap(lambda x: x.lower() if isinstance(x, str) else x)


C:\Users\ritik\AppData\Local\Temp\ipykernel_11128\4013432535.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  nbs_ = nbs_.applymap(lambda x: x.lower() if isinstance(x, str) else x)


In [8]:
nbs_['MultiModal_Label']=nbs_['MultiModal_Label'].apply(lambda x : 'biased' if x=='Likely' else 'non-biased')

In [9]:
nbs_['MultiModal_Label'] = nbs_['MultiModal_Label'].apply(lambda x: 1 if x == 'biased' else 0)

In [10]:
nbs_ = nbs_.iloc[5000:]

In [11]:
nbs_babe=nbs_[['headline','article_text','MultiModal_Label']]
nbs_babe.rename(columns={
    'article_text': 'article',
    'headline': 'text',
    'MultiModal_Label': 'label_bias' 
},inplace=True)
nbs_babe.columns

C:\Users\ritik\AppData\Local\Temp\ipykernel_11128\3516275186.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nbs_babe.rename(columns={


Index(['text', 'article', 'label_bias'], dtype='object')

In [12]:
nbs_nbs = nbs_[['caption','image_path','MultiModal_Label']]
nbs_nbs.rename(columns={
    'caption': 'article_text',
    'image_path': 'image_path',
    'MultiModal_Label': 'MultiModal_Label' 
},inplace = True)


C:\Users\ritik\AppData\Local\Temp\ipykernel_11128\3631541205.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nbs_nbs.rename(columns={


In [13]:
nbs_.columns

Index(['unique_id', 'outlet_x', 'title', 'content', 'text_label_x', 'outlet_y',
       'headline', 'top_image', 'article_text', 'new_categories',
       'news_categories_confidence_scores', 'text_label_y', 'image_filename',
       'image_path', 'text_labels', 'MultiModal_Label', 'caption'],
      dtype='object')

In [14]:
nbs_['caption']

5000    a woman in a blue jacket stands with a microph...
5001    an older man with white hair and a suit is see...
5002    a man in a suit stands in front of an american...
5003    a man in a blue suit stands in a room with a p...
5004    a man in a blue suit and red tie sits on a cha...
                              ...                        
6244    a black and white photo of a large assembly ha...
6245    an aerial view of a flooded road with a yellow...
6246    an individual stands in a barren landscape wit...
6247    a person in a fluffy red and white costume wit...
6248    an armored military vehicle with a mounted gun...
Name: caption, Length: 1249, dtype: object

In [15]:
nbs_nbs.columns

Index(['article_text', 'image_path', 'MultiModal_Label'], dtype='object')

In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


NBS plus Model that is text + image

Defining the model 

In [17]:

batch_size = 4 
epochs = 1
LEARNING_RATE = 3.2999999999999996e-05
fusion_type ='cosine_similarity'

In [18]:
class MultimodalClassifier(nn.Module):
    def __init__(
            self,
            text_encoder_id_or_path: str,
            image_encoder_id_or_path: str,
            projection_dim: int,
            fusion_method: Literal["concat", "align", "cosine_similarity"] = fusion_type,
            proj_dropout: float = 0.1,
            fusion_dropout: float = 0.1,
            num_classes: int = 2,
        ) -> None:
        super().__init__()

        self.fusion_method = fusion_method
        self.projection_dim = projection_dim
        self.num_classes = num_classes

        ##### Text Encoder
        self.text_encoder = BertModel.from_pretrained(text_encoder_id_or_path)
        self.text_projection = nn.Sequential(
            nn.Linear(self.text_encoder.config.hidden_size, self.projection_dim),
            nn.Dropout(proj_dropout),
        )

        ##### Image Encoder (using ResNet34 from AutoModel with timm)
        self.image_encoder = AutoModel.from_pretrained(image_encoder_id_or_path, trust_remote_code=True)
        self.image_encoder.classifier = nn.Identity()  # rm the classification head
        self.image_projection = nn.Sequential(
            nn.Linear(512, self.projection_dim),
            nn.Dropout(proj_dropout),
        )

        ##### Fusion Layer
        fusion_input_dim = self.projection_dim * 2 if fusion_method == "concat" else self.projection_dim
        self.fusion_layer = nn.Sequential(
            nn.Dropout(fusion_dropout),
            nn.Linear(fusion_input_dim, self.projection_dim),
            nn.GELU(),
            nn.Dropout(fusion_dropout),
        )

        ##### Classification Layer
        self.classifier = nn.Linear(self.projection_dim, self.num_classes)

    def forward(self, pixel_values: torch.Tensor, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        ##### Text Encoder Projection #####
        full_text_features = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True).last_hidden_state
        full_text_features = full_text_features[:, 0, :]  # using cls token
        full_text_features = self.text_projection(full_text_features)

        ##### Image Encoder Projection #####
        resnet_image_features = self.image_encoder(pixel_values=pixel_values).last_hidden_state
        
        # global average pooling for resent image features (bad idea? dim problems)
        resnet_image_features = resnet_image_features.mean(dim=[-2, -1])
        resnet_image_features = self.image_projection(resnet_image_features)

        ##### Fusion and Classification #####
        if self.fusion_method == "concat":
            fused_features = torch.cat([full_text_features, resnet_image_features], dim=-1)
        else:
            fused_features = full_text_features * resnet_image_features # don't think this works atm (should be dot prod)

        # fusion and classifier layers
        fused_features = self.fusion_layer(fused_features)
        classification_output = self.classifier(fused_features)

        return classification_output

def load_model():
    config = {
        "model_type": "multimodal-bias-classifier", 
        "text_encoder_id_or_path": "E:\\GitHub\\Multi-Model-Bias-Detection-and-Debiasing-the-News\\EDA\\Model_config", 
        "image_encoder_id_or_path": "resnet34", 
        "projection_dim": 768, 
        "fusion_method": "concat", 
        "num_classes": 1, "proj_dropout": 0.1, 
        "fusion_dropout": 0.1, "hidden_size": 768, 
        "save_components": ["resnet_encoder", "text_encoder", "fusion_layer", "classifier"], 
        "exclude_components": ["clip_text_encoder", "clip_image_encoder"]}


    model = MultimodalClassifier(
        text_encoder_id_or_path=config["text_encoder_id_or_path"],
        image_encoder_id_or_path="microsoft/resnet-34",
        projection_dim=config["projection_dim"],
        fusion_method=config["fusion_method"],
        proj_dropout=config["proj_dropout"],
        fusion_dropout=config["fusion_dropout"],
        num_classes=config["num_classes"]
    )

    model_weights_path = hf_hub_download(repo_id="maximuspowers/multimodal-bias-classifier", filename="model_weights.pth")
    checkpoint = torch.load(model_weights_path, map_location=torch.device('cpu'))
    model.load_state_dict(checkpoint, strict=False)

    return model


Loading the saved model

In [19]:
dict=torch.load("fine_tuned_model_nbs.pt")
model_NBS = load_model()
model_NBS.load_state_dict(dict)
model_NBS.eval()

Some weights of BertModel were not initialized from the model checkpoint at E:\GitHub\Multi-Model-Bias-Detection-and-Debiasing-the-News\EDA\Model_config and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


MultimodalClassifier(
  (text_encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-5): 6 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

Text Based Model (BABE Model)

Defining the base Model

In [20]:
from transformers import BertModel

class BertClass(torch.nn.Module):
    def __init__(self):
        super(BertClass, self).__init__()
        self.l1 = BertModel.from_pretrained("E:\\GitHub\\Multi-Model-Bias-Detection-and-Debiasing-the-News\\EDA\\Model_config")
        self.pre_classifier = torch.nn.Linear(768, 768)
        self.dropout = torch.nn.Dropout(0.1)
        self.classifier = torch.nn.Linear(768, 2)
        self.relu = torch.nn.ReLU()

    def forward(self, input_ids, attention_mask, token_type_ids):
        output_1 = self.l1(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        hidden_state = output_1[0]
        pooler = hidden_state[:, 0]
        pooler = self.pre_classifier(pooler)
        pooler = self.relu(pooler)
        pooler = self.dropout(pooler)
        output = self.classifier(pooler)
        return output


Loading the pre trained weights

In [21]:
dict=torch.load("BABE_fine_tuned_model.pt")
model_BABE = BertClass()
model_BABE.load_state_dict(dict)
model_BABE.eval()

Some weights of BertModel were not initialized from the model checkpoint at E:\GitHub\Multi-Model-Bias-Detection-and-Debiasing-the-News\EDA\Model_config and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertClass(
  (l1): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-5): 6 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=

In [22]:
import torch
from transformers import AutoTokenizer
from PIL import Image
import requests
from torchvision import transforms

Tokenizer

In [23]:
text_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
MAX_LEN = 512


Babe Dataset Data Loader and Dataset

In [24]:
from torch.utils.data import Dataset, DataLoader
import torch
class Dataloader_Babe(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.text1 = dataframe['article'].values
        self.text2 = dataframe['text'].values
        self.targets = dataframe['label_bias'].values
        self.max_len = max_len

    def __len__(self):
        return len(self.text1)

    def __getitem__(self, index):
        text1 = str(self.text1[index])
        text2 = str(self.text2[index])

        inputs = self.tokenizer(

            text1,
            text2,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_token_type_ids=True,
            return_attention_mask=True
        )
        ids = inputs['input_ids']
        mask = inputs['attention_mask']
        token_type_ids = inputs["token_type_ids"]


        return {
            'ids': torch.tensor(ids, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.long),
            'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long),
            'targets': torch.tensor(self.targets[index], dtype=torch.float)
        }
    

def train_ldr_for_babe(X_train,batch_size=2):
    
    training_set = Dataloader_Babe(X_train, text_tokenizer, MAX_LEN)
    train_params = {'batch_size': batch_size,
                'shuffle': True,
                'num_workers': 0
                }
    
    training_loader = DataLoader(training_set, **train_params)
    return training_loader


def valid_ldr_for_babe(X_test,batch_size=2):
    
    valid_set = Dataloader_Babe(X_test, text_tokenizer, MAX_LEN)
    valid_params = {'batch_size': batch_size,
                'shuffle': True,
                'num_workers': 0
                }
    
    valid_loader = DataLoader(valid_set, **valid_params)
    return valid_loader

NBS Plus: Data Loader 

In [25]:
nbs_nbs

,article_text,image_path,MultiModal_Label
5000,a woman in a blue jacket stands with a microph...,c:\users\ritik\downloads\images\images\18736ee...,0
5001,an older man with white hair and a suit is see...,c:\users\ritik\downloads\images\images\14c3316...,0
5002,a man in a suit stands in front of an american...,c:\users\ritik\downloads\images\images\16dacc5...,0
5003,a man in a blue suit stands in a room with a p...,c:\users\ritik\downloads\images\images\1b74984...,0
5004,a man in a blue suit and red tie sits on a cha...,c:\users\ritik\downloads\images\images\0753c02...,0
...,...,...,...
6244,a black and white photo of a large assembly ha...,c:\users\ritik\downloads\images\images\0471e10...,0
6245,an aerial view of a flooded road with a yellow...,c:\users\ritik\downloads\images\images\1e9e145...,0
6246,an individual stands in a barren landscape wit...,c:\users\ritik\downloads\images\images\0bc2d5d...,0
6247,a person in a fluffy red and white costume wit...,c:\users\ritik\downloads\images\images\1913a7c...,0


In [26]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class Dataloader_NBS_Plus(Dataset):
    def __init__(self, dataframe, text_tokenizer, image_transform):
        self.df = dataframe
        self.text_tokenizer = text_tokenizer
        self.image_transform = image_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row['article_text']
        image_path = row['image_path']

        # tokenize text
        text_inputs = self.text_tokenizer(
            text, padding='max_length', truncation=True, max_length=512 , return_tensors='pt'
        )
        # transform image
        image = Image.open(image_path).convert('RGB')
        image_input = self.image_transform(image)

        #label = torch.tensor(row['MultiModal_Label'])
        
        input_ids= text_inputs['input_ids'][0]
        attention_mask= text_inputs['attention_mask']
        pixel_values = image_input
        labels =  int(row['MultiModal_Label'])
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'pixel_values': image_input,
            'labels': torch.tensor(labels, dtype=torch.long)
        }

def train_dataloader_nbs(dataset):
    
    train_loader_dataset = Dataloader_NBS_Plus(dataset, text_tokenizer, image_transform)
    training_loader = DataLoader(train_loader_dataset, batch_size=4, shuffle=True)

    return training_loader


def valid_dataloader_nbs(dataset):
    
    test_loader_dataset = Dataloader_NBS_Plus(dataset, text_tokenizer, image_transform)

    val_loader = DataLoader(test_loader_dataset, batch_size=4, shuffle=True)
    return val_loader


Validation of Babe

In [27]:
from tqdm import tqdm

loss_function = torch.nn.CrossEntropyLoss()



def valid_BABE(model, testing_loader):
    prob_all = []
    prob_target = []
    model.eval()
    with torch.no_grad():
        for _, data in tqdm(enumerate(testing_loader, 0)):
            ids = data['ids'].to(device, dtype = torch.long)
            mask = data['mask'].to(device, dtype = torch.long)
            token_type_ids = data['token_type_ids'].to(device, dtype=torch.long)
            targets = data['targets'].to(device, dtype = torch.long)
            outputs = model(ids, mask, token_type_ids)
            prob=torch.softmax(outputs, dim=1)
            prob_all.append(prob)
            prob_target.append(targets)
            
    prob_all = torch.cat(prob_all).numpy()       
    prob_target = torch.cat(prob_target).numpy()
    return prob_all,prob_target

Validation of NBS

In [28]:
def valid_NBS(model, testing_loader):
    model.eval()
    prob_all = []
    prob_target = []
    with torch.no_grad():
        for _, data in tqdm(enumerate(testing_loader, 0)):
            ids = data['input_ids'].to(device, dtype = torch.long)
            mask = data['attention_mask'].to(device, dtype = torch.long)
            pixel_values = data['pixel_values'].to(device, dtype = torch.float)
            targets = data['labels'].reshape(-1,1).to(device, dtype = torch.float)

            outputs = model(input_ids=ids,
            attention_mask=mask,
            pixel_values=pixel_values)

            preds = torch.sigmoid(outputs)     
            prob = torch.sigmoid(outputs)
            prob_all.append(prob)
            prob_target.append(targets)
            

    prob_all = torch.cat(prob_all).numpy()       
    prob_target = torch.cat(prob_target).numpy()
    return prob_all,prob_target

#https://stackoverflow.com/questions/62301674/extracting-labels-after-applying-softmax  --> For Probability, Weights and Labels

In [29]:
from hyperopt import hp

space = {
    'w': hp.uniform('w', 0,1)
}

In [30]:
nbs_

,unique_id,outlet_x,title,content,text_label_x,outlet_y,headline,top_image,article_text,new_categories,news_categories_confidence_scores,text_label_y,image_filename,image_path,text_labels,MultiModal_Label,caption
5000,18736eea37,newsweek,"nikki haley mocks donald trump, joe biden with...",newsweek/images/18736eea37.txt,likely,newsweek,"nikki haley mocks donald trump, joe biden with...","{'bytes': none, 'path': '/projects/nmb-plus/co...",republican presidential hopeful nikki haley to...,['politics' 'national'],[0.95 0.85],likely,18736eea37.jpg,c:\users\ritik\downloads\images\images\18736ee...,biased,0,a woman in a blue jacket stands with a microph...
5001,14c3316900,newsweek,democrat warns u.s. response to jordan attack ...,newsweek/images/14c3316900.txt,likely,newsweek,democrat warns u.s. response to jordan attack ...,"{'bytes': none, 'path': '/projects/nmb-plus/co...",senator tim kaine said in an interview with cn...,['politics' 'international'],[0.9 0.8],likely,14c3316900.jpg,c:\users\ritik\downloads\images\images\14c3316...,biased,0,an older man with white hair and a suit is see...
5002,16dacc5669,newsweek,democrat brutally mocked for video of him sing...,newsweek/images/16dacc5669.txt,likely,newsweek,democrat brutally mocked for video of him sing...,"{'bytes': none, 'path': '/projects/nmb-plus/co...",colorado governor jared polis has been critici...,['politics' 'local/regional'],[0.95 0.85],likely,16dacc5669.jpg,c:\users\ritik\downloads\images\images\16dacc5...,biased,0,a man in a suit stands in front of an american...
5003,1b749844c0,newsweek,michael cohen's big 2024 plans - newsweek,newsweek/images/1b749844c0.txt,likely,newsweek,michael cohen's big 2024 plans - newsweek,"{'bytes': none, 'path': '/projects/nmb-plus/co...",former trump fixer michael cohen has big plans...,['politics' 'national'],[0.95 0.75],likely,1b749844c0.jpg,c:\users\ritik\downloads\images\images\1b74984...,biased,0,a man in a blue suit stands in a room with a p...
5004,0753c02b4a,newsweek,conservative slams maga for 'actually evil' co...,newsweek/images/0753c02b4a.txt,likely,newsweek,conservative slams maga for 'actually evil' co...,"{'bytes': none, 'path': '/projects/nmb-plus/co...",a conservative slammed make america great agai...,['politics'],[0.95],likely,0753c02b4a.jpg,c:\users\ritik\downloads\images\images\0753c02...,biased,0,a man in a blue suit and red tie sits on a cha...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6244,0471e108dd,cnn,opinion: we germans are making trump ‘thunders...,editor’s note: anna sauerbrey is the foreign e...,likely,cnn,opinion: we germans are making trump ‘thunders...,"{'bytes': none, 'path': '/projects/nmb-plus/wh...",editors note anna sauerbrey is the foreign edi...,['opinion/editorial' 'politics'],[0.9 0.8],likely,0471e108dd.jpg,c:\users\ritik\downloads\images\images\0471e10...,biased,0,a black and white photo of a large assembly ha...
6245,1e9e145f6e,cnn,brazil’s floods smashed through barriers desig...,karine pitana had just paid off the last insta...,likely,cnn,brazil’s floods smashed through barriers desig...,"{'bytes': none, 'path': '/projects/nmb-plus/wh...",karine pitana had just paid off the last insta...,['local/regional' 'national'],[0.85 0.75],likely,1e9e145f6e.jpg,c:\users\ritik\downloads\images\images\1e9e145...,biased,0,an aerial view of a flooded road with a yellow...
6246,0bc2d5dd25,cnn,"in a city cut off from the world, guns and dru...",on the rare days that the hills surrounding po...,likely,cnn,"in a city cut off from the world, guns and dru...","{'bytes': none, 'path': '/projects/nmb-plus/wh...",on the rare days that the hills surrounding po...,['national' 'international'],[0.8 0.7],likely,0bc2d5dd25.jpg,c:\users\ritik\downloads\images\images\0bc2d5d...,biased,0,an individual stands in a barren landscape wit...
6247,1913a7c1b2,cnn,switzerland wins eurovision after politically ...,switzerland’s nemo won a chaotic and political...,likely,cnn,switzerland wins eurovision 

In [31]:
'''
nbs_probs,true = valid_NBS(model_NBS,valid_dataloader_nbs(nbs_nbs))
babe_probs,true_ = valid_BABE(model_BABE,valid_ldr_for_babe(nbs_babe))
'''

'\nnbs_probs,true = valid_NBS(model_NBS,valid_dataloader_nbs(nbs_nbs))\nbabe_probs,true_ = valid_BABE(model_BABE,valid_ldr_for_babe(nbs_babe))\n'

In [32]:
'''nbs_probs.shape'''

'nbs_probs.shape'

In [33]:
'''babe_probs.shape'''

'babe_probs.shape'

In [34]:
'''babe_probs'''

'babe_probs'

In [35]:
'''nbs_probs'''

'nbs_probs'

In [36]:
from sklearn.metrics import f1_score
from hyperopt import STATUS_OK, STATUS_FAIL

def obj_func(params):
    try:
        

        
        nbs_probs,true = valid_NBS(model_NBS,valid_dataloader_nbs(nbs_nbs))
        babe_probs,true_ = valid_BABE(model_BABE,valid_ldr_for_babe(nbs_babe))

        w = params['w']
        total_rpob = w*nbs_probs.reshape(-1) + (1-w)*babe_probs[:,1]
        pred = (total_rpob > 0.5).astype(int)
        f1 = f1_score(true, pred, average='weighted')

        return {'loss': -f1, 'status': STATUS_OK}
    except Exception as e:
        print("Exception:", e)
        return {'status': STATUS_FAIL}

In [ ]:
from hyperopt import fmin, tpe, Trials

trials = Trials()

best = fmin(
    fn=obj_func,
    space=space,
    algo=tpe.suggest,
    max_evals=4,
    trials=trials
)

print("Best hyperparameters:", best)

  0%|          | 0/4 [00:00<?, ?trial/s, best loss=?]

0it [00:00, ?it/s]
C:\Users\ritik\AppData\Local\Temp\ipykernel_11128\247880750.py:33: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(input_ids, dtype=torch.long),

C:\Users\ritik\AppData\Local\Temp\ipykernel_11128\247880750.py:34: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'attention_mask': torch.tensor(attention_mask, dtype=torch.long),

1it [00:01,  1.52s/it]
2it [00:02,  1.43s/it]
3it [00:04,  1.36s/it]
4it [00:05,  1.23s/it]
5it [00:06,  1.16s/it]
6it [00:07,  1.12s/it]
7it [00:08,  1.13s/it]
8it [00:09,  1.14s/it]
9it [00:10,  1.20s/it]
10it [00:12,  1.27s/it]
11it [00:13,  1.36s/it]
12it [00:15,  1.36s/it]
13it [00:16,  1.32s/it]
14it [00:17,  1.27s/it]


 25%|██▌       | 1/4 [10:07<30:22, 607.42s/trial, best loss: -0.33132932531730125]

0it [00:00, ?it/s]
C:\Users\ritik\AppData\Local\Temp\ipykernel_11128\247880750.py:33: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(input_ids, dtype=torch.long),

C:\Users\ritik\AppData\Local\Temp\ipykernel_11128\247880750.py:34: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'attention_mask': torch.tensor(attention_mask, dtype=torch.long),

1it [00:01,  1.31s/it]
2it [00:02,  1.11s/it]
3it [00:03,  1.10s/it]
4it [00:04,  1.07s/it]
5it [00:05,  1.18s/it]
6it [00:06,  1.10s/it]
7it [00:07,  1.04s/it]
8it [00:08,  1.13s/it]
9it [00:09,  1.09s/it]
10it [00:11,  1.11s/it]
11it [00:12,  1.07s/it]
12it [00:13,  1.08s/it]
13it [00:14,  1.09s/it]
14it [00:15,  1.04s/it]


 50%|█████     | 2/4 [19:40<19:33, 586.94s/trial, best loss: -0.33132932531730125]

0it [00:00, ?it/s]
C:\Users\ritik\AppData\Local\Temp\ipykernel_11128\247880750.py:33: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(input_ids, dtype=torch.long),

C:\Users\ritik\AppData\Local\Temp\ipykernel_11128\247880750.py:34: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'attention_mask': torch.tensor(attention_mask, dtype=torch.long),

1it [00:00,  1.05it/s]
2it [00:01,  1.03it/s]
3it [00:03,  1.14s/it]
4it [00:04,  1.22s/it]
5it [00:05,  1.11s/it]
6it [00:06,  1.08s/it]
7it [00:07,  1.03s/it]
8it [00:08,  1.00s/it]
9it [00:09,  1.03s/it]
10it [00:10,  1.01s/it]
11it [00:11,  1.02s/it]
12it [00:12,  1.02s/it]
13it [00:13,  1.01it/s]
14it [00:14,  1.02s/it]


In [ ]:
class EnsembleModel(torch.nn.Module):
    def __init__(self,model_text_only,model_txt_and_img,valid_for_text,valid_for_imgandtxt,valid_loader_txt,valid_loader_textAndImage,w):
        super().__init__()
        ## Model Initialization
        self.model_text_only = model_text_only
        self.model_txt_and_img = model_txt_and_img

        #Parameter W for Fusion
        self.w = w

        #Dataloaders
        self.valid_loader_textAndImage = valid_loader_textAndImage
        self.valid_loader_txt = valid_loader_txt

        self.valid_for_text = valid_for_text
        self.valid_for_imgandtxt = valid_for_imgandtxt



    def forward(self,input_text,input_img_and_txt):
        probs_text,true = self.valid_for_text(self.model_text_only,self.valid_loader_txt(input_text))
        probs_textAndImage,true_ = self.valid_for_imgandtxt(self.model_txt_and_img,self.valid_loader_textAndImage(input_img_and_txt))
        total_rpob = self.w*probs_textAndImage.reshape(-1) + (1-self.w)*probs_text[:,1]
        pred = (total_rpob > 0.5).astype(int)

        return pred,true

: 